# Визуальная проверка bi-encoder retrieval: RoSBERTa ДО vs ПОСЛЕ дообучения

Для каждой из 19 размеченных пар (релевантный товар → пост) ноутбук рендерит HTML-блок:
- заголовок пары + ранг, на котором каждая модель нашла целевой пост (или «НЕ НАЙДЕНО»);
- описание товара (query);
- целевой пост;
- TOP-K кандидатов от **RoSBERTa base (до обучения)** — полные тексты, совпавший с целью — зелёный;
- TOP-K кандидатов от **RoSBERTa fine-tuned (после обучения)** — полные тексты, совпавший с целью — зелёный.

Тексты не обрезаются. Чистый bi-encoder, без cross-encoder.

---

**⚠ Важно.** Базовая и дообученная RoSBERTa имеют разные векторные пространства,
поэтому нужны **две разные таблицы** в LanceDB:
- `posts`       — индекс дообученной RoSBERTa (уже есть)
- `posts_base`  — индекс базовой `ai-forever/ru-en-RoSBERTa` (нужно создать отдельно)

Пока таблица `posts_base` не создана, ноутбук не запустится. Создать её можно тем же способом,
что и `posts3` (отдельный ноутбук на Colab для GPU-индексации 50k+ постов).


## 1. Импорты и настройки

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import json
import html
import torch
import lancedb
from sentence_transformers import SentenceTransformer
from IPython.display import HTML, display

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")


In [ ]:
# ==================== НАСТРОЙКИ ====================
BI_ENCODER_BASE_PATH = "ai-forever/ru-en-RoSBERTa"      # до обучения (из HF)
BI_ENCODER_FT_PATH   = "models/final/bi-encoder"        # после обучения (локально)

LANCEDB_PATH     = "./lancedb_store"
TABLE_BASE       = "rosberta-base-50k"   # таблица для базовой RoSBERTa (нужно создать)
TABLE_FT         = "rosberta-fine-tuned-50k"        # таблица для дообученной RoSBERTa

GT_POSTS_JSON = "ground_truth_posts.json"
GT_PAIRS_JSON = "ground_truth_pairs.json"

TOP_K = 20  # сколько постов показывать на каждый запрос


## 2. Ground truth, модели, базы

In [ ]:
with open(GT_POSTS_JSON, encoding='utf-8') as f:
    gt_posts = json.load(f)
with open(GT_PAIRS_JSON, encoding='utf-8') as f:
    gt_pairs = json.load(f)

post_text_by_num = {p['post_id']: p['text'] for p in gt_posts}

print(f'GT постов: {len(gt_posts)}')
print(f'GT пар (товар → пост): {len(gt_pairs)}')


In [ ]:
print('Загружаем RoSBERTa base (до обучения)...')
bi_base = SentenceTransformer(BI_ENCODER_BASE_PATH, device=device)
print(f'  dim={bi_base.get_sentence_embedding_dimension()}')

print('Загружаем RoSBERTa fine-tuned (после обучения)...')
bi_ft = SentenceTransformer(BI_ENCODER_FT_PATH, device=device)
print(f'  dim={bi_ft.get_sentence_embedding_dimension()}')

db = lancedb.connect(LANCEDB_PATH)
table_base = db.open_table(TABLE_BASE)
table_ft   = db.open_table(TABLE_FT)
print(f'\nТаблица {TABLE_BASE}: {table_base.count_rows():,} записей')
print(f'Таблица {TABLE_FT}: {table_ft.count_rows():,} записей')


## 3. Поиск и HTML-рендер

In [ ]:
def encode_base(q):
    # RoSBERTa — без префиксов
    return bi_base.encode([q], normalize_embeddings=True)[0].tolist()

def encode_ft(q):
    # RoSBERTa — без префиксов
    return bi_ft.encode([q], normalize_embeddings=True)[0].tolist()

def search(table, qvec, k):
    return (table.search(qvec, query_type='vector')
                 .limit(k)
                 .select(['text', 'channel', 'category'])
                 .to_list())

def esc(s):
    return html.escape(str(s)).replace('\n', '<br>')

def find_rank(results, target_text):
    t = target_text.strip()
    for i, r in enumerate(results, 1):
        if r['text'].strip() == t:
            return i
    return None

def rank_badge(rank, label, bg_color):
    if rank is None:
        return (f'<span style="background:#f8d7da;color:#000;padding:6px 12px;'
                f'border-radius:4px;font-weight:bold;border:1px solid #f1aeb5;">'
                f'{label}: не найдено в TOP-{TOP_K}</span>')
    return (f'<span style="background:{bg_color};color:#000;padding:6px 12px;'
            f'border-radius:4px;font-weight:bold;border:1px solid #adb5bd;">'
            f'{label}: ранг #{rank}</span>')

def card(rank, post, is_target, accent_bg):
    if is_target:
        bg = '#d4edda'; border = '#28a745'; badge_bg = '#c3e6cb'
        star = ' ★ ЦЕЛЬ'
    else:
        bg = '#f8f9fa'; border = '#dee2e6'; badge_bg = accent_bg
        star = ''
    return (
        f'<div style="background:{bg};border-left:4px solid {border};'
        f'padding:12px 14px;margin:8px 0;border-radius:4px;color:#000;">'
        f'<div style="margin-bottom:8px;font-size:13px;color:#000;">'
        f'<span style="background:{badge_bg};color:#000;padding:3px 9px;'
        f'border-radius:3px;font-weight:bold;border:1px solid #adb5bd;">#{rank}</span>'
        f'<span style="color:#000;margin-left:10px;">'
        f'<b>@{esc(post["channel"])}</b> · {esc(post.get("category",""))}'
        f'{star}</span></div>'
        f'<div style="white-space:pre-wrap;font-family:system-ui,sans-serif;'
        f'font-size:14px;color:#000;line-height:1.5;">{esc(post["text"])}</div>'
        f'</div>'
    )

def render_pair(idx, total, pair, target_text, res_base, res_ft):
    rank_base = find_rank(res_base, target_text)
    rank_ft   = find_rank(res_ft, target_text)

    cards_base = ''.join(
        card(i, r, r['text'].strip() == target_text.strip(), '#e2d9f3')
        for i, r in enumerate(res_base, 1)
    )
    cards_ft = ''.join(
        card(i, r, r['text'].strip() == target_text.strip(), '#cfe2ff')
        for i, r in enumerate(res_ft, 1)
    )

    return HTML(f'''
    <div style="border:2px solid #495057;border-radius:8px;margin:28px 0;
                background:#ffffff;font-family:system-ui,sans-serif;
                overflow:hidden;color:#000;">

        <div style="background:#e9ecef;color:#000;padding:14px 20px;
                    border-bottom:1px solid #ced4da;">
            <div style="font-size:18px;font-weight:bold;color:#000;">
                Пара {idx}/{total} · GT post #{pair["post_num"]}
            </div>
            <div style="font-size:13px;color:#000;margin-top:4px;">
                {esc(pair.get("imt_name",""))} · {esc(pair.get("subj_name",""))}
            </div>
        </div>

        <div style="padding:16px 20px;background:#f8f9fa;
                    display:flex;gap:14px;flex-wrap:wrap;
                    border-bottom:1px solid #e9ecef;">
            {rank_badge(rank_base, "RoSBERTa base",      "#e2d9f3")}
            {rank_badge(rank_ft,   "RoSBERTa fine-tuned", "#cfe2ff")}
        </div>

        <div style="padding:20px;background:#ffffff;color:#000;">
            <div style="background:#e7f3ff;border-left:4px solid #0d6efd;
                        padding:12px 14px;margin-bottom:14px;border-radius:4px;color:#000;">
                <div style="font-weight:bold;color:#000;margin-bottom:6px;
                            font-size:13px;text-transform:uppercase;letter-spacing:0.5px;">
                    Запрос (описание товара)
                </div>
                <div style="white-space:pre-wrap;font-size:14px;line-height:1.5;color:#000;">
                    {esc(pair["description"])}
                </div>
            </div>

            <div style="background:#d4edda;border-left:4px solid #28a745;
                        padding:12px 14px;margin-bottom:20px;border-radius:4px;color:#000;">
                <div style="font-weight:bold;color:#000;margin-bottom:6px;
                            font-size:13px;text-transform:uppercase;letter-spacing:0.5px;">
                    Целевой пост (должен найтись)
                </div>
                <div style="white-space:pre-wrap;font-size:14px;line-height:1.5;color:#000;">
                    {esc(target_text)}
                </div>
            </div>

            <h4 style="color:#000;border-bottom:2px solid #6f42c1;
                       padding-bottom:6px;margin:24px 0 8px 0;">
                RoSBERTa base (до обучения) — TOP {TOP_K}
            </h4>
            {cards_base}

            <h4 style="color:#000;border-bottom:2px solid #0d6efd;
                       padding-bottom:6px;margin:28px 0 8px 0;">
                RoSBERTa fine-tuned (после обучения) — TOP {TOP_K}
            </h4>
            {cards_ft}
        </div>
    </div>
    ''')


## 4. Сводка: где нашёлся целевой пост по каждой паре

In [ ]:
# Сначала быстрая сводная таблица: ранги по всем парам
summary_rows = []
for idx, pair in enumerate(gt_pairs, 1):
    target_text = post_text_by_num[pair['post_num']]

    qvec_base = encode_base(pair['description'])
    qvec_ft   = encode_ft(pair['description'])
    res_base  = search(table_base, qvec_base, TOP_K)
    res_ft    = search(table_ft,   qvec_ft,   TOP_K)

    summary_rows.append({
        'idx': idx,
        'pair': pair,
        'target_text': target_text,
        'res_base': res_base,
        'res_ft': res_ft,
        'rank_base': find_rank(res_base, target_text),
        'rank_ft':   find_rank(res_ft,   target_text),
    })

# HTML-сводка
def fmt_rank(r):
    if r is None:
        return ('<span style="background:#f8d7da;color:#000;padding:2px 8px;'
                'border-radius:3px;font-weight:bold;border:1px solid #f1aeb5;">—</span>')
    if r <= 5:
        bg = '#d4edda'; bd = '#28a745'
    elif r <= 20:
        bg = '#fff3cd'; bd = '#ffc107'
    else:
        bg = '#f8d7da'; bd = '#f1aeb5'
    return (f'<span style="background:{bg};color:#000;padding:2px 8px;'
            f'border-radius:3px;font-weight:bold;border:1px solid {bd};">#{r}</span>')

rows_html = ''.join(
    f'<tr style="background:#ffffff;color:#000;">'
    f'<td style="padding:6px 10px;color:#000;border-bottom:1px solid #e9ecef;">{row["idx"]}</td>'
    f'<td style="padding:6px 10px;color:#000;border-bottom:1px solid #e9ecef;">GT #{row["pair"]["post_num"]}</td>'
    f'<td style="padding:6px 10px;color:#000;border-bottom:1px solid #e9ecef;">{esc(row["pair"].get("imt_name",""))[:70]}</td>'
    f'<td style="padding:6px 10px;text-align:center;border-bottom:1px solid #e9ecef;">{fmt_rank(row["rank_base"])}</td>'
    f'<td style="padding:6px 10px;text-align:center;border-bottom:1px solid #e9ecef;">{fmt_rank(row["rank_ft"])}</td></tr>'
    for row in summary_rows
)

hits_base = sum(1 for r in summary_rows if r['rank_base'] is not None)
hits_ft   = sum(1 for r in summary_rows if r['rank_ft']   is not None)

display(HTML(f'''
<div style="font-family:system-ui,sans-serif;margin:16px 0;color:#000;">
    <div style="font-size:16px;margin-bottom:10px;color:#000;">
        <b>Hit rate в TOP-{TOP_K}:</b>
        RoSBERTa base <span style="color:#000;font-weight:bold;background:#e2d9f3;padding:2px 8px;border-radius:3px;">{hits_base}/{len(summary_rows)}</span>,
        RoSBERTa fine-tuned <span style="color:#000;font-weight:bold;background:#cfe2ff;padding:2px 8px;border-radius:3px;">{hits_ft}/{len(summary_rows)}</span>
    </div>
    <table style="border-collapse:collapse;font-size:13px;width:100%;background:#ffffff;color:#000;">
        <thead>
            <tr style="background:#e9ecef;color:#000;">
                <th style="padding:8px 10px;text-align:left;color:#000;border-bottom:2px solid #adb5bd;">#</th>
                <th style="padding:8px 10px;text-align:left;color:#000;border-bottom:2px solid #adb5bd;">GT пост</th>
                <th style="padding:8px 10px;text-align:left;color:#000;border-bottom:2px solid #adb5bd;">Товар</th>
                <th style="padding:8px 10px;color:#000;border-bottom:2px solid #adb5bd;">base</th>
                <th style="padding:8px 10px;color:#000;border-bottom:2px solid #adb5bd;">fine-tuned</th>
            </tr>
        </thead>
        <tbody>
            {rows_html}
        </tbody>
    </table>
    <div style="font-size:11px;color:#000;margin-top:6px;">
        Зелёный — ранг ≤5, жёлтый — ≤20, красный — не найдено в TOP-{TOP_K}
    </div>
</div>
'''))


## 5. Подробный просмотр всех пар

Проскролль вниз, читай глазами. Совпавший с целью пост выделен зелёным.

In [ ]:
for row in summary_rows:
    display(render_pair(
        row['idx'], len(summary_rows), row['pair'],
        row['target_text'], row['res_base'], row['res_ft']
    ))
